# 01 · Vector add in CuPy RawKernel - _speaker demo_

**Goal:** see what a GPU kernel actually is - a string of CUDA C, JIT-compiled and launched from Python.

This is the only CuPy moment in the workshop. Everything after this is in Triton.

## Environment bootstrap

In [ ]:
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try: importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
ensure('cupy-cuda12x', 'cupy')
import cupy as cp
assert cp.cuda.runtime.getDeviceCount() > 0, 'No CUDA GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.'
print(f'cupy {cp.__version__} | gpu {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')


## The kernel - CUDA C as a Python string

Everything between the triple quotes is real CUDA C compiled at runtime. Things to notice:

- `__global__` marks a function callable from the host (Python) and run on the GPU.
- `blockIdx.x * blockDim.x + threadIdx.x` is the universal “where am I in the grid?” idiom.
- The `if (i < n)` guard is why kernels need explicit masking - we will see Triton's `mask=` equivalent next.

In [ ]:
kernel_src = r'''
extern "C" __global__
void vec_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
'''

vec_add = cp.RawKernel(kernel_src, 'vec_add')

## Inputs

In [ ]:
n = 1 << 20  # one million elements
a = cp.random.random(n, dtype=cp.float32)
b = cp.random.random(n, dtype=cp.float32)
c = cp.empty_like(a)
print(f'n = {n:,} elements')

## Launch

In [ ]:
threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
cp.cuda.runtime.deviceSynchronize()
print(f'launched {blocks_per_grid:,} blocks x {threads_per_block} threads = {blocks_per_grid * threads_per_block:,} threads total')

## Correctness

In [ ]:
cp.testing.assert_allclose(c, a + b)
print('correctness: ok')

## Timing (a quick measurement)

In [ ]:
start = cp.cuda.Event(); end = cp.cuda.Event()
# warmup
for _ in range(3):
    vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
cp.cuda.runtime.deviceSynchronize()

start.record()
for _ in range(100):
    vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
end.record()
end.synchronize()
ms = cp.cuda.get_elapsed_time(start, end) / 100
bandwidth_gbs = (3 * a.nbytes) / (ms * 1e-3) / 1e9  # read a, read b, write c
print(f'avg kernel time: {ms:.3f} ms')
print(f'effective bandwidth: {bandwidth_gbs:.1f} GB/s')

## Teaching beats

1. The CUDA C string is the **only** place anything truly “GPU” exists. Everything else is launcher glue.
2. `blockIdx.x * blockDim.x + threadIdx.x` is the universal “where am I?” idiom - circle it twice.
3. The `if (i < n)` guard is why kernels need explicit masking. We will see Triton's `mask=` equivalent in the next notebook.
4. Compare `kernel((blocks,), (threads,), ...)` to the equivalent CPU loop - same algorithm, different launcher.

On to Triton.